# Vignette 6: Evaluating synthetic image quality

The paper measured quality with a physician reader study: whether an image looked
machine-generated and whether it was consistent with its label. Those judgements correlated with
classifier gains. This notebook computes proxies that need no physicians, for any synthetic
source, and exports a stratified sample in the reader-study format so the physician study can
be repeated.

1. **Distribution distance** between synthetic and real images in CLIP and DINOv2 embedding
   space (Fréchet distance and kernel distance, the FID/KID formulas applied to those features).
2. **Label consistency**: a classifier trained on real images only, applied to the synthetic
   images; the fraction it assigns to the intended label.
3. **Memorization**: nearest real neighbour similarity.
4. **Reader-study export**: `survey.csv`, anonymized images, `answer_key.csv`.

In [ ]:
SMALL = True
SOURCES = {"sd2.1-dreambooth-2024": None}   # name -> generation folder; None = 2024 HF text-to-image shards
PER_LABEL = 64
MAX_EPOCHS = 8 if SMALL else 50
READER_STUDY_PER_LABEL = 3 if SMALL else 7

In [ ]:
import os, json, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

os.environ.setdefault("HF_HOME", "/workspace/.cache/huggingface")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # less fragmentation on a 32 GB card
WORK = Path(os.environ.get("SYNDERM_WORK_DIR", "/workspace/synderm"))
WORK.mkdir(parents=True, exist_ok=True)
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

def show_grid(rows, row_titles=None, col_titles=None, size=2.6, suptitle=None):
    """rows: list of lists of PIL images (one list per row)."""
    n_rows, n_cols = len(rows), max(len(r) for r in rows)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(size * n_cols, size * n_rows), squeeze=False)
    for r, row in enumerate(rows):
        for c in range(n_cols):
            ax = axes[r][c]; ax.axis("off")
            if c < len(row) and row[c] is not None:
                ax.imshow(row[c])
            if r == 0 and col_titles and c < len(col_titles):
                ax.set_title(col_titles[c], fontsize=10)
        if row_titles:
            axes[r][0].text(-0.08, 0.5, row_titles[r], transform=axes[r][0].transAxes, rotation=90,
                            va="center", ha="right", fontsize=10)
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout(); plt.show()

In [ ]:
from synderm.data import load_sample, load_2024_shards, SyntheticDataset
real_train, real_val = load_sample(split="train"), load_sample(split="val")
classes = real_train.labels
sources = {}
for name, folder in SOURCES.items():
    sources[name] = SyntheticDataset(folder, labels=classes) if folder else load_2024_shards(
        classes, method="finetune-text-to-image", submethod="text-to-image", max_per_label=PER_LABEL,
        exclude_ref_ids=set(real_val.df["md5hash"]))
{k: len(v) for k, v in sources.items()}

## 1. Distribution distances

Lower is closer to the real distribution. Values are only comparable between sources computed
on the same real reference set and the same embedding model.

In [ ]:
from synderm.eval import ImageEmbedder, embed_dataset, frechet_distance, kernel_distance, nearest_real_neighbors
from synderm.eval.embeddings import CLIP_MODEL, DINOV2_MODEL
rows = []
feats_cache = {}
for emb_name in [CLIP_MODEL, DINOV2_MODEL]:
    emb = ImageEmbedder(emb_name)
    real_f, _ = embed_dataset(real_val, emb)
    train_f, _ = embed_dataset(real_train, emb)
    rows.append({"embedding": emb_name, "source": "real train vs real held-out (reference)",
                 "frechet": frechet_distance(train_f, real_f), "kid": kernel_distance(train_f, real_f, subset_size=min(100, len(real_f)))[0]})
    for name, ds in sources.items():
        f, meta = embed_dataset(ds, emb)
        feats_cache[(emb_name, name)] = (f, meta)
        sim, _ = nearest_real_neighbors(train_f, f)
        rows.append({"embedding": emb_name, "source": name, "frechet": frechet_distance(f, real_f),
                     "kid": kernel_distance(f, real_f, subset_size=min(100, len(real_f)))[0],
                     "max_sim_to_train": float(sim.max()), "frac_sim_gt_0.95": float((sim > 0.95).mean())})
    del emb; torch.cuda.empty_cache()
pd.DataFrame(rows).round(4)

## 2. Label consistency

A classifier trained on real images only is applied to each synthetic source. The agreement
rate is a stand-in for the reader study's "consistent with the label" question.

In [ ]:
from synderm.classify import ClassificationDataset, build_transforms, train_classifier, evaluate
from torch.utils.data import DataLoader
from synderm.utils import synthetic_train_val_split
train, val = synthetic_train_val_split(real_train, None, per_class_test_size=8, random_state=0)
judge = train_classifier(ClassificationDataset(train, classes, build_transforms(augment=True)), ClassificationDataset(val, classes),
                         classes, test_dataset=ClassificationDataset(real_val, classes), max_epochs=MAX_EPOCHS, verbose=False)
print(f"judge accuracy on real held-out: {judge.test_metrics['accuracy']:.3f}")
consistency = {}
for name, ds in sources.items():
    m = evaluate(judge.model, DataLoader(ClassificationDataset(ds, classes), batch_size=64, num_workers=4), classes)
    consistency[name] = {"overall": m["accuracy"], **{c: v["accuracy"] for c, v in m["per_class"].items()}}
pd.DataFrame(consistency).round(3)

## 3. Reader-study export

The paper showed physicians 7 real, 7 in-then-outpaint and 7 text-to-image images per label in
random order with two questions. `export_reader_study` writes the same layout.

In [ ]:
from synderm.eval import export_reader_study
study_dir = WORK / "vignettes" / "06_reader_study"
survey = export_reader_study({"real": real_val, **sources}, study_dir, classes, per_label_per_source=READER_STUDY_PER_LABEL)
print(len(survey), "images ->", study_dir)
survey.head()